# Precious metals capstone workflow

This notebook orchestrates the same functions as the CLI. Inputs and outputs are local. Add verified COT release evidence before running.

In [1]:
from pathlib import Path
import os

project_root = next((
    path for path in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
    if (path / "pyproject.toml").is_file() and (path / "config/config.yml").is_file()
), None)
if project_root is None:
    raise FileNotFoundError("Could not find the project root (pyproject.toml and config/config.yml) from the notebook working directory.")
os.chdir(project_root)

from capstone.cli import read_config, execute
from capstone.data import refresh_sources
config = read_config("config/config.yml", "config/settings.yml")
config

{'data_dir': 'data',
 'release_manifest': 'config/release_dates.csv',
 'start': '2018-01-01',
 'end': None,
 'target_symbols': ['GLD', 'SLV'],
 'required_symbols': ['GLD', 'SLV', 'BTC-USD', 'NVDA'],
 'symbols': {'GLD': 'GLD',
  'SLV': 'SLV',
  'BTC-USD': 'BTC-USD',
  'SOL-USD': 'SOL-USD',
  'ETH-USD': 'ETH-USD',
  'ZEC-USD': 'ZEC-USD',
  'NVDA': 'NVDA',
  '^GSPC': '^GSPC',
  '^IXIC': '^IXIC',
  '^DJI': '^DJI'},
 'market_sources': {'BTC-USD': {'provider': 'binance', 'ticker': 'BTCUSDT'},
  'SOL-USD': {'provider': 'binance',
   'ticker': 'SOLUSDT',
   'history_start': '2020-08-11'},
  'ETH-USD': {'provider': 'binance', 'ticker': 'ETHUSDT'},
  'ZEC-USD': {'provider': 'binance',
   'ticker': 'ZECUSDT',
   'history_start': '2019-03-21'}},
 'market_features': {'BTC-USD': 'btc',
  'SOL-USD': 'sol',
  'ETH-USD': 'eth',
  'ZEC-USD': 'zec',
  'NVDA': 'nvda',
  '^GSPC': 'sp500',
  '^IXIC': 'nasdaq',
  '^DJI': 'dow'},
 'cot_max_age_days': 7,
 'holding_sessions': 5,
 'cross_market_max_age_days': 4,

## Optional refresh
Run this cell when network access is available. It refreshes local snapshots and can overwrite existing files. Alternatively, run `make data` from the project root. CFTC release evidence must be maintained independently.

In [2]:
import logging

logger = logging.getLogger("capstone.workflow")
logger.setLevel(logging.INFO)
if not logger.handlers:
    handler = logging.StreamHandler()
    handler.setFormatter(logging.Formatter("%(asctime)s %(levelname)s %(message)s", "%H:%M:%S"))
    logger.addHandler(handler)
logger.propagate = False

logger.info("Refreshing configured market and CFTC source snapshots")
try:
    refresh_failures = refresh_sources(config)
except Exception:
    logger.exception("Source refresh failed")
    raise

if refresh_failures:
    logger.warning("Refresh completed with unavailable optional sources: %s", ", ".join(sorted(refresh_failures)))
else:
    logger.info("Source refresh completed successfully")
refresh_failures

08:32:07 INFO Refreshing configured market and CFTC source snapshots
08:32:11 INFO Source refresh completed successfully


{}

## Offline workflow
This calls the same functions as the command line: load verified snapshots, derive features, select models, simulate and report. It requires local snapshots and their hash metadata under `data/`; it does not download missing inputs. If the next cell reports a missing snapshot, run the refresh cell above or `make data` when source access is available, then rerun this cell.

In [3]:
from IPython.display import Markdown, display

try:
    execute(config, refresh=False, stage="run")
except RuntimeError as exc:
    message = str(exc)
    if message.startswith("Required market data ") and "unavailable: Missing cached data or hash metadata:" in message:
        display(Markdown(
            "### Workflow not run: required market snapshot is missing\n\n"
            f"`{message}`\n\n"
            "Run the optional refresh cell when network access is available, then rerun this cell. "
            "No features, model metrics, or backtest results were produced."
        ))
    else:
        raise
except FileNotFoundError as exc:
    message = str(exc)
    if "Missing cached data or hash metadata:" in message:
        display(Markdown(
            "### Workflow not run: required source snapshot is missing\n\n"
            f"`{message}`\n\n"
            "Run the optional refresh cell when network access is available, then rerun this cell. "
            "No features, model metrics, or backtest results were produced."
        ))
    else:
        raise
except ValueError as exc:
    message = str(exc)
    expected_data_limits = (
        "At least 40 labeled session dates are required",
        "Insufficient observations after purging overlapping labels",
    )
    if message in expected_data_limits:
        display(Markdown(
            "### Workflow not run: insufficient labeled history\n\n"
            f"`{message}`\n\n"
            "The cached sources do not provide enough usable labeled sessions for the configured "
            "chronological splits. No model or backtest results were produced. Check source coverage "
            "and COT release evidence before treating a refreshed snapshot as research-ready."
        ))
    else:
        raise
else:
    import pandas as pd

    display(Markdown(
        "### Model comparison on validation\n\n"
        "Each row is fitted on purged training data and scored on validation. "
        "`selected=True` marks the minimum validation MAE; ties keep candidate order. "
        "MAE/RMSE are decimal returns (0.01 = 1%). "
        "Estimator parameters are recorded as JSON; holdout scores are separate."
    ))
    display(pd.read_csv(Path(config.get("output_dir", "artifacts")) / "model_comparison.csv"))


Model comparison: validation only; selected by minimum MAE (returns in decimal units).
   model      split      mae     rmse             estimator                                                                                                                                                                                                                                                                                                                                                                                              parameters  selected
    mean validation 0.036398 0.058627        DummyRegressor                                                                                                                                                                                                                                                                                                                                                {"constant": null, "quantile": null, "strategy": "mean"}

### Model comparison on validation

Each row is fitted on purged training data and scored on validation. `selected=True` marks the minimum validation MAE; ties keep candidate order. MAE/RMSE are decimal returns (0.01 = 1%). Estimator parameters are recorded as JSON; holdout scores are separate.

,model,split,mae,rmse,estimator,parameters,selected
0,mean,validation,0.036398,0.058627,DummyRegressor,"{""constant"": null, ""quantile"": null, ""strategy...",True
1,tree_3,validation,0.036436,0.058636,DecisionTreeRegressor,"{""ccp_alpha"": 0.0, ""criterion"": ""squared_error...",False
2,forest_3,validation,0.036521,0.058663,RandomForestRegressor,"{""bootstrap"": true, ""ccp_alpha"": 0.0, ""criteri...",False
3,forest_5,validation,0.036744,0.058890,RandomForestRegressor,"{""bootstrap"": true, ""ccp_alpha"": 0.0, ""criteri...",False
4,tree_5,validation,0.036779,0.058901,DecisionTreeRegressor,"{""ccp_alpha"": 0.0, ""criterion"": ""squared_error...",False
5,ridge_10,validation,0.039805,0.060961,Ridge,"{""alpha"": 10.0, ""copy_X"": true, ""fit_intercept...",False
6,ridge_1,validation,0.040596,0.061560,Ridge,"{""alpha"": 1.0, ""copy_X"": true, ""fit_intercept""...",False
